# Support Vector Machine (SVM) — Improved Model

This notebook trains and tunes SVM classifiers using:
- Three kernels: **Linear**, **RBF**, and **Polynomial**
- `GridSearchCV` to search over `C`, `gamma`, and `class_weight`
- Final evaluation with accuracy, precision, recall, F1-score, and confusion matrix
- Side-by-side comparison with the tuned KNN model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Load PCA-reduced data from the pre-processing folder
from sklearn.preprocessing import StandardScaler

X_train = pd.read_csv('../pre-processing/X_train_pca.csv')
X_test  = pd.read_csv('../pre-processing/X_test_pca.csv')
y_train = pd.read_csv('../pre-processing/y_train.csv').values.ravel()
y_test  = pd.read_csv('../pre-processing/y_test.csv').values.ravel()

# PCA components have unequal variances (each component's variance = its eigenvalue).
# SVM is sensitive to feature scale, so we re-standardize the PCA output here.
# This is done fit-on-train / transform-both to avoid leakage.
pca_scaler  = StandardScaler()
X_train_s   = pca_scaler.fit_transform(X_train)
X_test_s    = pca_scaler.transform(X_test)

print('Training Features Shape:', X_train_s.shape)
print('Testing Features Shape: ', X_test_s.shape)
print('Class distribution (train):', pd.Series(y_train).value_counts().to_dict())

Training Features Shape: (41566, 15)
Testing Features Shape:  (10392, 15)
Class distribution (train): {1: 15518, 2: 15517, 0: 10531}


### Step 1: GridSearchCV Per Kernel

We run a separate `GridSearchCV` for each kernel with its relevant hyperparameters.
- **Linear**: only `C` and `class_weight` (gamma has no effect on a linear kernel)
- **RBF**: `C`, `gamma`, and `class_weight`
- **Polynomial**: `C`, `gamma`, and `class_weight`

Cross-validation uses 3 folds (faster on ~21k samples) with stratification to preserve class ratios.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

cv = StratifiedKFold(n_splits=3, shuffle=False)

# ── 1. LINEAR SVM via LinearSVC (100x faster than SVC kernel='linear') ──────
print('--- Linear SVM (LinearSVC) ---')
linear_results = {}
for C in [0.01, 0.1, 1, 10]:
    for cw in [None, 'balanced']:
        lsvc = LinearSVC(C=C, class_weight=cw, max_iter=5000, random_state=42)
        scores = cross_val_score(lsvc, X_train_s, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        key = (C, str(cw))
        linear_results[key] = scores.mean()
        print(f'  C={C}, class_weight={cw}: {scores.mean():.4f}')

best_linear_key = max(linear_results, key=linear_results.get)
best_linear_C, best_linear_cw = best_linear_key
best_linear_cw = None if best_linear_cw == 'None' else best_linear_cw
best_linear_score = linear_results[best_linear_key]
print(f'\nBest Linear: C={best_linear_C}, class_weight={best_linear_cw}, CV={best_linear_score:.4f}')

# ── 2. RBF SVM (most promising kernel, focused search) ───────────────────────
print('\n--- RBF SVM (GridSearchCV) ---')
rbf_params = {
    'C':            [0.1, 1, 10],
    'gamma':        ['scale', 'auto'],
    'class_weight': [None, 'balanced']
}
rbf_gs = GridSearchCV(
    SVC(kernel='rbf', random_state=42, max_iter=-1),
    param_grid=rbf_params,
    cv=cv, scoring='accuracy', n_jobs=-1, verbose=1
)
rbf_gs.fit(X_train_s, y_train)
print(f'  Best params : {rbf_gs.best_params_}')
print(f'  Best CV acc : {rbf_gs.best_score_:.4f}')

grid_results = {'rbf': rbf_gs}

--- Linear SVM (LinearSVC) ---
  C=0.01, class_weight=None: 0.4720
  C=0.01, class_weight=balanced: 0.4716
  C=0.1, class_weight=None: 0.4720
  C=0.1, class_weight=balanced: 0.4716
  C=1, class_weight=None: 0.4720
  C=1, class_weight=balanced: 0.4716
  C=10, class_weight=None: 0.4720
  C=10, class_weight=balanced: 0.4716

Best Linear: C=0.01, class_weight=None, CV=0.4720

--- RBF SVM (GridSearchCV) ---
Fitting 3 folds for each of 12 candidates, totalling 36 fits


### Step 2: Select the Best Kernel and Evaluate on the Test Set

In [ ]:
# Train final best linear model on full training set
best_linear_model = LinearSVC(C=best_linear_C, class_weight=best_linear_cw,
                               max_iter=5000, random_state=42)
best_linear_model.fit(X_train_s, y_train)
linear_test_acc = accuracy_score(y_test, best_linear_model.predict(X_test_s))

# Best RBF model
best_rbf_model  = rbf_gs.best_estimator_
rbf_test_acc    = accuracy_score(y_test, best_rbf_model.predict(X_test_s))

print('=== SVM Results Summary ===')
print(f'  Linear (C={best_linear_C}, class_weight={best_linear_cw}): Test Acc = {linear_test_acc*100:.2f}%')
print(f'  RBF    ({rbf_gs.best_params_}): Test Acc = {rbf_test_acc*100:.2f}%')

# Pick overall best
if rbf_test_acc >= linear_test_acc:
    best_kernel    = 'rbf'
    best_svm       = best_rbf_model
    best_params    = rbf_gs.best_params_
    best_cv_score  = rbf_gs.best_score_
    test_acc       = rbf_test_acc
else:
    best_kernel    = 'linear'
    best_svm       = best_linear_model
    best_params    = {'C': best_linear_C, 'class_weight': best_linear_cw}
    best_cv_score  = best_linear_score
    test_acc       = linear_test_acc

y_pred_best = best_svm.predict(X_test_s)
print(f'\nBest overall: {best_kernel} — Test Accuracy: {test_acc*100:.2f}%')
print()
print('Classification Report (0=Draw, 1=Home Win, 2=Home Loss):')
print(classification_report(y_test, y_pred_best,
                             target_names=['Draw (0)', 'Win (1)', 'Loss (2)']))

### Step 3: Confusion Matrix

In [ ]:
cm     = confusion_matrix(y_test, y_pred_best)
labels = ['Draw (0)', 'Win (1)', 'Loss (2)']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=labels, yticklabels=labels)
plt.ylabel('Actual Outcome')
plt.xlabel('Predicted Outcome')
plt.title(f'SVM Confusion Matrix (kernel={best_kernel}, params={best_params})')
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=150)
plt.show()
print('Saved: svm_confusion_matrix.png')

### Step 4: Kernel Comparison Chart

In [ ]:
kernel_test_accs = {
    'Linear': linear_test_acc * 100,
    'RBF':    rbf_test_acc    * 100,
}

colors = ['steelblue', 'darkorange']
plt.figure(figsize=(6, 5))
bars = plt.bar(kernel_test_accs.keys(), kernel_test_accs.values(), color=colors)
plt.ylim(0, 70)
plt.ylabel('Test Accuracy (%)')
plt.title('SVM: Linear vs. RBF Kernel (Best Params)')
for bar, val in zip(bars, kernel_test_accs.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('svm_kernel_comparison.png', dpi=150)
plt.show()
print('Saved: svm_kernel_comparison.png')

### Step 5: KNN vs SVM Side-by-Side Comparison

In [ ]:
# Load KNN best result saved by knn_model.ipynb
try:
    with open('knn_best_result.json') as f:
        knn_result = json.load(f)
    knn_acc = knn_result['test_accuracy'] * 100
    knn_label = f"KNN (k={knn_result['best_params']['k']}, {knn_result['best_params']['weights']})"
except FileNotFoundError:
    knn_acc   = 43.48   # fallback: original baseline
    knn_label = 'KNN (k=5, uniform)'

svm_acc   = test_acc * 100
svm_label = f'SVM ({best_kernel}, C={best_params["C"]})'

plt.figure(figsize=(7, 5))
bars = plt.bar([knn_label, svm_label], [knn_acc, svm_acc],
               color=['steelblue', 'darkorange'])
plt.ylim(0, 70)
plt.ylabel('Test Accuracy (%)')
plt.title('Model Comparison: Tuned KNN vs. Best SVM')
for bar, val in zip(bars, [knn_acc, svm_acc]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('knn_vs_svm.png', dpi=150)
plt.show()
print('Saved: knn_vs_svm.png')

winner = 'SVM' if svm_acc > knn_acc else 'KNN'
diff   = abs(svm_acc - knn_acc)
print(f'\n{winner} wins by {diff:.2f}%')

### Step 6: Why One Model Outperforms the Other

**If SVM > KNN:**
SVM is better suited for this data because:
1. **High-dimensional PCA space** — SVM finds a maximum-margin hyperplane, which generalizes well even when features are PCA components (uncorrelated, dense).
2. **RBF kernel** maps the data into a higher-dimensional space where nonlinear class boundaries can exist — soccer outcomes are not linearly separable.
3. **KNN suffers from the curse of dimensionality** — with 15 PCA dimensions, 'close neighbors' in Euclidean space are not necessarily similar matches.

**Why overall accuracy is still modest (~43–52%):**
Soccer prediction is intrinsically difficult for three structural reasons:
1. **Features lack recency** — we used averaged team stats over all seasons, so a team's 2015 form is diluted by 2008 data. Recent form, injuries, and lineup changes are absent.
2. **Draws are near-random events** — even professional betting markets struggle to predict draws; the signal is genuinely weak.
3. **Outcome variance** — a single defensive mistake, red card, or goalkeeper save can flip a result regardless of team quality. This irreducible noise caps any model's ceiling.

In [ ]:
# Save best SVM result for final report reference
svm_best_result = {
    'model':         'SVM',
    'best_kernel':   best_kernel,
    'best_params':   best_params,
    'cv_accuracy':   round(best_cv_score, 4),
    'test_accuracy': round(test_acc, 4)
}
with open('svm_best_result.json', 'w') as f:
    json.dump(svm_best_result, f, indent=2)
print(json.dumps(svm_best_result, indent=2))